# Setup Environment & Repository

In [ ]:
from pathlib import Path
import os
import subprocess

REPOSITORY_URL = "https://github.com/Omid-Nejati/Locality-iN-Locality.git"
repository_candidates = [
    Path.cwd(),
    Path.cwd() / "Locality-iN-Locality-main",
    Path.cwd() / "Locality-iN-Locality",
]
repository_root = next(
    (path for path in repository_candidates if (path / "LNL_MoEx.py").exists()),
    None,
)

if repository_root is None:
    repository_root = Path.cwd() / "Locality-iN-Locality"
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(repository_root)],
        check=True,
    )

os.chdir(repository_root)
print("Working directory:", Path.cwd())

%pip install -q timm==0.9.16 einops scikit-learn


# Imports & Config

In [ ]:
import os
import math
import random
import shutil
from pathlib import Path
import csv
from urllib.request import urlretrieve
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

from PIL import Image
from torch.utils.data import DataLoader
import timm

SEED = 42

def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("timm:", timm.__version__)
print("Device:", device)


# Download & Prepare Dataset

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
GTSRB_URLS = {
    "GTSRB_Final_Training_Images.zip": (
        "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/"
        "GTSRB_Final_Training_Images.zip"
    ),
    "GTSRB_Final_Test_Images.zip": (
        "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/"
        "GTSRB_Final_Test_Images.zip"
    ),
    "GTSRB_Final_Test_GT.zip": (
        "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/"
        "GTSRB_Final_Test_GT.zip"
    ),
}

for filename, url in GTSRB_URLS.items():
    destination = DATA_DIR / filename
    if not destination.exists():
        print("Downloading", filename)
        urlretrieve(url, destination)
    else:
        print("Already downloaded", filename)

dataset_extracted = (DATA_DIR / "GTSRB").exists() and (DATA_DIR / "GT-final_test.csv").exists()
if not dataset_extracted:
    for filename in GTSRB_URLS:
        with zipfile.ZipFile(DATA_DIR / filename) as archive:
            archive.extractall(DATA_DIR)
    print("GTSRB archives extracted.")
else:
    print("GTSRB dataset already extracted.")

gtsrb_dir = DATA_DIR / "GTSRB"
test_images_dir = gtsrb_dir / "Final_Test" / "Images"
test_dir = gtsrb_dir / "test"
test_dir.mkdir(parents=True, exist_ok=True)

# ImageFolder needs the official test images arranged by class directory.
with (DATA_DIR / "GT-final_test.csv").open(newline="") as csv_file:
    for row in csv.DictReader(csv_file, delimiter=";"):
        class_dir = test_dir / f"{int(row['ClassId']):04d}"
        class_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(test_images_dir / row["Filename"], class_dir)


# Data Loader Setup

In [ ]:
class ResizeWithPadding:
    """Resize while preserving aspect ratio, then pad to a square image."""

    def __init__(self, size: int = 224, fill: int = 128):
        self.size = size
        self.fill = fill

    def __call__(self, image: Image.Image) -> Image.Image:
        width, height = image.size
        scale = self.size / max(width, height)

        new_width = max(1, round(width * scale))
        new_height = max(1, round(height * scale))

        image = TF.resize(
            image,
            [new_height, new_width],
            interpolation=transforms.InterpolationMode.BICUBIC,
            antialias=True,
        )

        horizontal_padding = self.size - new_width
        vertical_padding = self.size - new_height

        left = horizontal_padding // 2
        right = horizontal_padding - left
        top = vertical_padding // 2
        bottom = vertical_padding - top

        return TF.pad(
            image,
            [left, top, right, bottom],
            fill=self.fill,
            padding_mode="constant",
        )

NORMALIZE_MEAN = (0.5, 0.5, 0.5)
NORMALIZE_STD = (0.5, 0.5, 0.5)

eval_transform = transforms.Compose([
    ResizeWithPadding(size=224, fill=128),
    transforms.ToTensor(),
    transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])

TEST_ROOT = DATA_DIR / "GTSRB" / "test"
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 2

testset = torchvision.datasets.ImageFolder(
    TEST_ROOT,
    transform=eval_transform,
)

test_loader = DataLoader(
    testset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
    persistent_workers=NUM_WORKERS > 0,
    drop_last=False,
)

NUM_CLASSES = len(testset.classes)
print("Test images:", len(testset))
print("Classes:", NUM_CLASSES)


# Model & Evaluation Function

In [ ]:
from LNL_MoEx import LNL_MoEx_Ti

@torch.inference_mode()
def calculate_top1(model: nn.Module, loader: DataLoader) -> float:
    """Evaluate clean Top-1 accuracy without retaining gradients."""
    model.eval()

    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
            logits = model(images)

        predictions = logits.argmax(dim=1)
        correct += predictions.eq(labels).sum().item()
        total += labels.size(0)

    return 100.0 * correct / total

def _load_torch_checkpoint(path: Path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:  # Compatibility with older PyTorch versions.
        return torch.load(path, map_location=device)


# Checkpoint Evaluation

Please set `CHECKPOINT_PATH` to the location of your trained PyTorch checkpoint (.pth).

In [ ]:
# TODO: Specify your checkpoint path here
# For example: CHECKPOINT_PATH = Path("checkpoints/LNL_MoEx_RandAug_Ti_GTSRB_fresh_v4_ema_warmstart_seed42_top1.pth")
CHECKPOINT_PATH = Path("./checkpoints/your_checkpoint.pth")
TARGET_VALIDATION_TOP1 = 99.5

if not CHECKPOINT_PATH.exists():
    print(f"[ERROR] Checkpoint not found at: {CHECKPOINT_PATH}")
    print("Please provide a valid checkpoint path.")
else:
    print(f"Loading checkpoint from: {CHECKPOINT_PATH}")
    
    verification_model = LNL_MoEx_Ti(
        pretrained=False,
        num_classes=NUM_CLASSES,
    ).to(device)
    
    checkpoint = _load_torch_checkpoint(CHECKPOINT_PATH)
    
    # Try to load best EMA model first, fallback to standard EMA, then fallback to model_state_dict
    state_dict = checkpoint.get("best_ema_model_state_dict")
    if state_dict is None:
        state_dict = checkpoint.get(
            "ema_model_state_dict",
            checkpoint.get("model_state_dict", checkpoint),
        )
    
    # Load state dict
    verification_model.load_state_dict(state_dict, strict=True)
    
    print("Running evaluation on test set...")
    test_top1 = calculate_top1(
        verification_model,
        test_loader,
    )
    
    print(f"Top-1 accuracy: {test_top1:.3f}%")
    if test_top1 > TARGET_VALIDATION_TOP1:
        print(f"Target > {TARGET_VALIDATION_TOP1:.1f}%: achieved.")
    else:
        print(f"Target > {TARGET_VALIDATION_TOP1:.1f}%: not achieved yet.")
